# From Naive to Sophisticated: A Programming Progression

This notebook traces five versions of the same program — printing stanzas of the nursery rhyme *The House that Jack Built* — each one using a more advanced programming concept than the last.

The rhyme is **cumulative**: every new stanza repeats all previous phrases and adds one more. That structure makes it a perfect vehicle for exploring loops, recursion, lists, and functions, because the problem itself has a natural recursive shape.

---
**The rhyme (first five stanzas):**

> This is the house that Jack built.
>
> This is the malt that lay in the house that Jack built.
>
> This is the rat that ate the malt that lay in the house that Jack built.
>
> This is the cat that killed the rat that ate the malt that lay in the house that Jack built.
>
> This is the dog that worried the cat that killed the rat that ate the malt that lay in the house that Jack built.

---

As you read through each version, ask yourself: *what did we gain, and what did we give up, at each step?*

## Version 1 — The Naive Approach: Just Print Everything

The simplest possible solution: one `print()` call per line, nothing more.

A programmer who only knows `print()` would write this. It works perfectly, and that matters — **correctness always comes first**. But notice the problems:

- Every phrase is typed out in full, over and over. Adding a sixth stanza means a lot of copy-paste.
- If you want to change a phrase (say, fix a typo in "the house that Jack built"), you must find and fix every occurrence.
- The code has no structure that reflects the structure of the rhyme itself.

This version scores high on **readability** but low on **maintainability**.

In [ ]:
# Every stanza is written out by hand.
# Nothing is reused; every phrase is repeated in full.

print("This is the house that Jack built.")
print()

print("This is the malt that lay in")
print("the house that Jack built.")
print()

print("This is the rat that ate the malt that lay in")
print("the house that Jack built.")
print()

print("This is the cat that killed the rat that ate the malt that lay in")
print("the house that Jack built.")
print()

print("This is the dog that worried the cat that killed the rat that ate the malt that lay in")
print("the house that Jack built.")

## Version 2 — Using a Loop

The key insight here is that each stanza differs from the previous one only in *how many phrases* it includes. That's a pattern a loop can express.

We use the **loop variable** `stanza` as a threshold: a phrase is printed in stanza `n` only if `n` is large enough to include it. No data structures are needed — just `if` conditions inside a `for` loop.

What we gain:
- Adding a sixth stanza means adding one `if` condition and incrementing the range — two small changes instead of rewriting a whole block.
- The blank line between stanzas is handled automatically.

What we still lack:
- The phrases are still written as string literals scattered through the code. They are not stored anywhere reusable.
- The logic and the data are tangled together.

In [ ]:
# The outer loop counts stanzas: stanza 1, 2, 3, 4, 5.

STANZAS_TO_PRINT = 5
# empty string: ""
# single space: " "

for stanza in range(1, STANZAS_TO_PRINT+1):

    # Every stanza starts with "This is".
    # end=" " keeps the cursor on the same line so the next print continues there.
    print("This is", end=" ")

    # Each phrase is only printed if the current stanza number is high enough.
    # The conditions are ordered newest-to-oldest so the output reads correctly.
    if stanza >= 5:
        print("the dog that worried", end=" ")
    if stanza >= 4:
        print("the cat that killed", end=" ")
    if stanza >= 3:
        print("the rat that ate", end=" ")
    if stanza >= 2:
        print("the malt that lay in", end=" ")

    # The anchor phrase ends every stanza; no condition needed.
    print("the house that Jack built.")

    # Blank line between stanzas (skipped after the last one would be cleaner,
    # but this is intentionally kept simple).
    print()

: 

## Version 3 — Using a Recursive Function

The rhyme has a naturally recursive shape: *stanza N = new phrase + stanza N-1*. A recursive function expresses that structure directly in code.

`recite(n)` prints phrase `n`, then calls `recite(n-1)`, which prints phrase `n-1`, and so on, until the **base case** `n == 1` prints the anchor phrase and stops.

What we gain:
- The code mirrors the mathematical structure of the problem — each call is one step of unwinding.
- The `for` loop in the driver is now trivially simple.

What to watch out for:
- Recursion requires a base case. Without `return` at `n == 1`, the function would call `recite(0)`, then `recite(-1)`, forever.
- The phrases are still hardcoded inside the function — data and logic are still mixed.

In [ ]:
def recite(n):
    """
    Print the tail of stanza n: phrase n, then phrase n-1, ..., down to phrase 1.
    Each recursive call handles one phrase and delegates the rest to itself.
    """
    # Base case: phrase 1 is the anchor that ends every stanza.
    # We print it with a newline and stop recursing.
    if n == 1:
        print("the house that Jack built.")
        return

    # Recursive cases: print this stanza's new phrase (no newline yet),
    # then hand off to recite(n-1) to continue the chain.
    if n == 2:
        print("the malt that lay in", end=" ")
    elif n == 3:
        print("the rat that ate", end=" ")
    elif n == 4:
        print("the cat that killed", end=" ")
    elif n == 5:
        print("the dog that worried", end=" ")

    recite(n - 1)  # Print all the phrases that came before this one.


# Driver: print each stanza by calling recite() with increasing depth.
for stanza in range(1, 6):
    print("This is", end=" ")  # Every stanza begins with "This is".
    recite(stanza)
    print()

## Version 4 — Using a List

Now we separate **data** from **logic** for the first time. The phrases live in a list; the loop just indexes into it.

The list is ordered from innermost to outermost — `phrases[0]` is the anchor, `phrases[1]` is the next layer, and so on. The inner loop walks **backward** from the current stanza index down to 0, which is exactly the order the rhyme needs.

What we gain:
- To add a sixth stanza, you add one string to the list. The loops need no changes at all.
- Fixing a typo means editing one string in one place.
- `range(len(phrases))` makes the code length-agnostic: it works for any number of phrases.

What we still lack:
- The list is a global, and the loops are at the top level. Everything is in one flat block with no organisation.
- There is no way to reuse or test part of this code in isolation.

In [ ]:
# All phrases in one place, ordered from innermost (anchor) to outermost.
# Changing or extending the poem means editing this list — nothing else.
phrases = [
    "the house that Jack built.",   # index 0 — the anchor, ends every stanza
    "the malt that lay in",          # index 1
    "the rat that ate",              # index 2
    "the cat that killed",           # index 3
    "the dog that worried",          # index 4
]

# Outer loop: one iteration per stanza.
# range(len(phrases)) works regardless of how many phrases are in the list.
for stanza in range(len(phrases)):
    print("This is", end=" ")

    # Inner loop: walk backward from the current phrase down to the anchor.
    # stanza=0 prints only phrases[0]; stanza=1 prints phrases[1] then phrases[0]; etc.
    for i in range(stanza, -1, -1):
        print(phrases[i], end=" ")

    print()   # End the stanza line.
    print()   # Blank line between stanzas.

## Version 5 — Functions, Separation of Concerns, and a `main()` Guard

This is the version a professional would write. The improvements are organisational, not algorithmic — the output is identical to Version 4.

**Three principles at work:**

1. **Separation of concerns.** `build_stanza` knows how to *construct* a stanza string. `print_poem` knows how to *display* the poem. `main` owns the data and *orchestrates* the other two. None of them does the other's job.

2. **Pure functions.** `build_stanza` takes inputs and returns a value with no side effects — it does not print anything. This means you can call it in a test, inspect its return value, or reuse it in a web app or a file writer without touching the print logic.

3. **The `if __name__ == "__main__"` guard.** When Python imports a module, it runs all top-level code immediately. The guard ensures that `main()` only runs when the file is *executed directly*, not when it is imported by another module. This is a near-universal Python convention.

**What `" ".join(...)` does:** instead of printing each phrase with `end=" "`, we collect all the phrases for a stanza into a sequence and join them with a single space. This is more Pythonic and avoids a trailing space at the end of the line.

In [ ]:
def build_stanza(phrases, n):
    """
    Return the full text of stanza n as a single string.

    This is a pure function: it takes inputs, returns a value,
    and does not print or modify anything outside itself.
    """
    # Walk backward from phrases[n] to phrases[0] and join with spaces.
    # "This is " is prepended to complete the sentence.
    return "This is " + " ".join(phrases[i] for i in range(n, -1, -1))


def print_poem(phrases):
    """
    Print all stanzas of the poem, one per line, separated by blank lines.

    This function only handles presentation — it delegates stanza
    construction entirely to build_stanza.
    """
    for n in range(len(phrases)):
        print(build_stanza(phrases, n))
        print()


def main():
    """
    Entry point: define the data and run the program.

    Keeping the data here (rather than as a global) means it is easy
    to find, change, or replace — for example, with a different rhyme.
    """
    phrases = [
        "the house that Jack built.",
        "the malt that lay in",
        "the rat that ate",
        "the cat that killed",
        "the dog that worried",
    ]
    print_poem(phrases)


# In a .py file this guard prevents main() from running on import.
# In a notebook every cell is run directly, so this always executes — 
# but the habit of writing it is worth building early.
if __name__ == "__main__":
    main()

## Summary

| Version | Concept introduced | Key benefit |
|---|---|---|
| 1 — Naive | `print()` only | Simplest possible; easy to read |
| 2 — Loop | `for` loop + `if` thresholds | Eliminates repetition; easy to extend |
| 3 — Recursion | Recursive function | Code structure mirrors problem structure |
| 4 — List | List + index arithmetic | Data and logic separated; one edit extends the poem |
| 5 — Functions | Pure functions, `main()`, `__name__` guard | Testable, reusable, professionally organised |

Each version is a valid program. The progression is not about making the code *work better* — the output is the same every time. It is about making the code **easier to read, change, test, and reuse** as programs grow larger and more complex.